# 05 — Final Load Prep & KPI Computation

Compute the KPI framework promised in the problem statement and export the Tableau-ready dataset plus a set of pre-aggregated summary tables used directly in the dashboard.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
CLEAN_PATH         = PROJECT_ROOT / 'data' / 'processed' / 'cleaned_dataset.csv'
TABLEAU_READY_PATH = PROJECT_ROOT / 'data' / 'processed' / 'tableau_ready_dataset.csv'
KPI_DIR            = PROJECT_ROOT / 'data' / 'processed' / 'kpi'
KPI_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CLEAN_PATH, parse_dates=['start_time', 'end_time'])
print(f'Loaded {len(df):,} rows × {df.shape[1]} columns')

Loaded 449,691 rows × 48 columns


## KPI framework

| KPI | Definition | Government use |
|---|---|---|
| Severity Index by State | mean(severity) per state | Prioritise state-level interventions |
| Accident Density by City | count per city | Identify hotspot cities |
| Weather Risk Score | % severity ≥ 3 per weather condition | Trigger weather advisories |
| Rush-Hour Severity Rate | mean severity in rush vs non-rush | Staff emergency response |
| Infrastructure Risk Flag | % high-severity per POI type | Prioritise infrastructure upgrades |
| YoY Growth Rate | (year - prev year) / prev year | Track programme effectiveness |
| Nighttime Severity Premium | mean severity night - day | Justify lighting investments |

### KPI 1 — Severity index by state

In [2]:
state_kpi = (
    df.groupby('state').agg(
        accident_count=('severity', 'size'),
        severity_index=('severity', 'mean'),
        high_severity_rate=('is_high_severity', 'mean'),
        avg_duration_min=('duration_min', 'mean'),
    )
    .round(3)
    .sort_values('severity_index', ascending=False)
)
state_kpi['high_severity_rate'] = (state_kpi['high_severity_rate'] * 100).round(2)
state_kpi.to_csv(KPI_DIR / 'kpi_state.csv')
state_kpi.head(10)

,accident_count,severity_index,high_severity_rate,avg_duration_min
state,,,,
GA,10099,2.533,46.7,92.890
VT,50,2.520,46.0,108.766
WI,2034,2.501,40.2,118.577
RI,1129,2.488,49.1,89.025
IA,1562,2.453,38.0,102.640
MO,4391,2.450,41.5,109.025
CO,5403,2.448,39.0,88.204
KY,2034,2.440,41.5,83.894
IN,4097,2.414,33.9,101.948


### KPI 2 — Accident density by city (top 100)

In [3]:
city_kpi = (
    df.groupby(['state', 'city']).agg(
        accident_count=('severity', 'size'),
        severity_index=('severity', 'mean'),
        high_severity_rate=('is_high_severity', 'mean'),
    )
    .round(3)
    .sort_values('accident_count', ascending=False)
    .head(100)
)
city_kpi['high_severity_rate'] = (city_kpi['high_severity_rate'] * 100).round(2)
city_kpi.to_csv(KPI_DIR / 'kpi_city_top100.csv')
city_kpi.head(10)

accident_count  severity_index  high_severity_rate
state city                                                           
TX    Houston               10281           2.191                19.0
FL    Miami                 10112           2.118                10.6
CA    Los Angeles            9376           2.253                24.8
NC    Charlotte              8409           2.047                 6.2
TX    Dallas                 7765           2.282                28.4
      Austin                 5959           2.093                 9.6
FL    Orlando                5920           2.079                 7.5
NC    Raleigh                5113           2.102                11.3
TN    Nashville              4397           2.153                16.5
LA    Baton Rouge            4197           2.086                 9.7

### KPI 3 — Weather risk score

In [4]:
weather_kpi = (
    df.groupby('weather_condition').agg(
        accident_count=('severity', 'size'),
        severity_index=('severity', 'mean'),
        weather_risk_score=('is_high_severity', 'mean'),
    )
    .round(3)
)
weather_kpi['weather_risk_score'] = (weather_kpi['weather_risk_score'] * 100).round(2)
weather_kpi = weather_kpi[weather_kpi['accident_count'] >= 1000].sort_values('weather_risk_score', ascending=False)
weather_kpi.to_csv(KPI_DIR / 'kpi_weather.csv')
weather_kpi.head(15)

,accident_count,severity_index,weather_risk_score
weather_condition,,,
Overcast,24712,2.389,35.3
Scattered Clouds,13162,2.379,35.2
Clear,52178,2.366,33.5
Heavy Rain,1769,2.299,28.7
Rain,4813,2.291,27.1
Light Rain,20831,2.268,24.8
Unknown,9944,2.281,24.5
Light Snow,7136,2.274,23.6
Light Drizzle,1343,2.253,23.5


### KPI 4 — Rush-hour severity rate

In [5]:
rush_kpi = (
    df.groupby('is_rush_hour').agg(
        accident_count=('severity', 'size'),
        severity_index=('severity', 'mean'),
        high_severity_rate=('is_high_severity', 'mean'),
    )
    .round(3)
)
rush_kpi['high_severity_rate'] = (rush_kpi['high_severity_rate'] * 100).round(2)
rush_kpi.index = rush_kpi.index.map({True: 'rush_hour', False: 'non_rush'})
rush_kpi.to_csv(KPI_DIR / 'kpi_rush_hour.csv')
rush_kpi

,accident_count,severity_index,high_severity_rate
is_rush_hour,,,
non_rush,248730,2.234,21.1
rush_hour,200961,2.224,21.6


### KPI 5 — Infrastructure risk flag

In [6]:
poi_cols = ['traffic_signal', 'junction', 'crossing', 'stop', 'railway',
            'station', 'amenity', 'bump', 'roundabout', 'give_way', 'no_exit', 'traffic_calming']
poi_cols = [c for c in poi_cols if c in df.columns]

rows = []
for col in poi_cols:
    present = df[df[col]]
    rows.append({
        'poi': col,
        'accident_count': len(present),
        'severity_index': round(present['severity'].mean(), 3) if len(present) else np.nan,
        'high_severity_rate_pct': round(present['is_high_severity'].mean() * 100, 2) if len(present) else np.nan,
    })
poi_kpi = pd.DataFrame(rows).sort_values('high_severity_rate_pct', ascending=False)
poi_kpi.to_csv(KPI_DIR / 'kpi_poi.csv', index=False)
poi_kpi

,poi,accident_count,severity_index,high_severity_rate_pct
1,junction,33638,2.323,29.27
4,railway,3926,2.174,16.66
9,give_way,2244,2.170,15.69
11,traffic_calming,410,2.159,14.39
10,no_exit,1125,2.125,12.62
7,bump,189,2.127,12.17
0,traffic_signal,70419,2.090,9.75
5,station,11665,2.082,8.36
2,crossing,52511,2.065,7.40
6,amenity,5659,2.078,7.02


### KPI 6 — Year-over-year growth rate

In [7]:
yearly = df.groupby('year').agg(
    accident_count=('severity', 'size'),
    severity_index=('severity', 'mean'),
).round(3)
yearly['yoy_growth_pct'] = (yearly['accident_count'].pct_change() * 100).round(2)
yearly.to_csv(KPI_DIR / 'kpi_yoy.csv')
yearly

,accident_count,severity_index,yoy_growth_pct
year,,,
2016,26288,2.374,NaN
2017,46629,2.388,77.38
2018,57988,2.388,24.36
2019,61517,2.309,6.09
2020,74562,2.188,21.21
2021,90289,2.141,21.09
2022,81576,2.079,-9.65
2023,10842,2.058,-86.71


### KPI 7 — Nighttime severity premium

In [8]:
if 'sunrise_sunset' in df.columns:
    daynight = df.groupby('sunrise_sunset').agg(
        accident_count=('severity', 'size'),
        severity_index=('severity', 'mean'),
        high_severity_rate=('is_high_severity', 'mean'),
    ).round(3)
    daynight['high_severity_rate'] = (daynight['high_severity_rate'] * 100).round(2)
    daynight.to_csv(KPI_DIR / 'kpi_day_night.csv')
    print(daynight)
    if {'Day', 'Night'}.issubset(set(daynight.index)):
        premium = daynight.loc['Night', 'severity_index'] - daynight.loc['Day', 'severity_index']
        print(f'\nNighttime severity premium: {premium:+.3f} severity points')

                accident_count  severity_index  high_severity_rate
sunrise_sunset                                                    
Day                     311781           2.227                21.7
Night                   136810           2.235                20.6
Unknown                   1100           2.231                13.8

Nighttime severity premium: +0.008 severity points


## Tableau-ready export

Re-shape the row-level dataset with only the columns the Tableau workbook actually needs. Trimming unused columns keeps the workbook performant on Tableau Public (which has a 15M-row and 10 GB extract limit).

In [9]:
tableau_cols = [
    # IDs & geography
    'state', 'city', 'county', 'zipcode', 'timezone',
    'start_lat', 'start_lng',
    # time
    'start_time', 'end_time', 'date', 'year', 'month', 'month_name',
    'day_of_week', 'day_name', 'hour', 'season', 'is_weekend', 'is_rush_hour',
    # severity
    'severity', 'severity_label', 'is_high_severity', 'duration_min', 'distance_mi',
    # weather
    'weather_condition', 'temperature_f', 'humidity', 'visibility_mi',
    'wind_speed_mph', 'precipitation_in', 'pressure_in',
    'sunrise_sunset', 'civil_twilight',
    # infrastructure POI
    'traffic_signal', 'junction', 'crossing', 'stop', 'railway',
    'station', 'amenity', 'bump', 'roundabout', 'give_way', 'no_exit', 'traffic_calming',
]
tableau_cols = [c for c in tableau_cols if c in df.columns]

tableau_df = df[tableau_cols].copy()
print(f'Tableau dataset: {len(tableau_df):,} rows × {tableau_df.shape[1]} columns')

Tableau dataset: 449,691 rows × 45 columns


In [10]:
TABLEAU_READY_PATH.parent.mkdir(parents=True, exist_ok=True)
tableau_df.to_csv(TABLEAU_READY_PATH, index=False)
print(f'✓ Saved Tableau-ready dataset → {TABLEAU_READY_PATH}')
print(f'  Size on disk: {TABLEAU_READY_PATH.stat().st_size / 1e6:.1f} MB')

✓ Saved Tableau-ready dataset → /Users/aryankinha/Downloads/B_G14_DVACapstone-main/data/processed/tableau_ready_dataset.csv
  Size on disk: 140.8 MB


## Output summary

Files written to `data/processed/`:

- `cleaned_dataset.csv` — row-level cleaned output (from notebook 02)
- `tableau_ready_dataset.csv` — trimmed row-level file for Tableau
- `kpi/kpi_state.csv`
- `kpi/kpi_city_top100.csv`
- `kpi/kpi_weather.csv`
- `kpi/kpi_rush_hour.csv`
- `kpi/kpi_poi.csv`
- `kpi/kpi_yoy.csv`
- `kpi/kpi_day_night.csv`

All KPI files can be connected in Tableau as supplementary data sources for the executive-view sheets, while `tableau_ready_dataset.csv` drives operational drill-downs and map views.